# Dimensionality Reduction And Clustering

**REQUIRED DAY 2**

## Load your checkpoint

Fresh kernel -- loading back the `adata` saved at the end of [06_normalization_and_feature_selection.ipynb](06_normalization_and_feature_selection.ipynb) (normalized, log-transformed, highly-variable genes flagged).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_06_normalized.h5ad")
adata


## PCA, neighbors, UMAP

A cell-by-gene matrix with thousands of genes is too high-dimensional to cluster or visualize directly. The standard pipeline compresses it in stages:

In [ ]:
sc.pp.pca(adata, n_comps=50)               # compress to the top principal components
sc.pp.neighbors(adata)                      # build a graph of each cell's nearest neighbors in PCA space
sc.tl.umap(adata)                           # 2D layout for visualization only — not used for clustering itself
sc.pl.umap(adata)

PCA finds the axes of greatest variation; the neighbor graph captures which cells are transcriptionally similar; UMAP is a 2D projection for *looking at* that structure — it's a visualization, not the thing clustering actually operates on.

## Clustering, and the parameter nobody can tell you the "right" value for

In [ ]:
sc.tl.leiden(adata, resolution=1.0, key_added="leiden")
sc.pl.umap(adata, color="leiden")

`resolution` controls how fine-grained the clusters are — higher resolution means more, smaller clusters. **There is no universally correct resolution.** Try two different values on the same data:

In [ ]:
sc.tl.leiden(adata, resolution=0.4, key_added="leiden_coarse")
sc.tl.leiden(adata, resolution=1.5, key_added="leiden_fine")
sc.pl.umap(adata, color=["leiden_coarse", "leiden_fine"])

Both are "valid" clusterings of the same data — they answer different questions (broad cell classes vs. finer subtypes). The number itself isn't defensible; **the resolution you end up using should be justified against something concrete**, like whether clusters separate along known marker genes (checked in [08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb)), not left at whatever a tutorial's default happened to be. This is Agent-B checklist item 15.

## The other thing a cluster boundary might mean: not biology

If today's data had multiple lanes, batches, or donors, a cluster boundary that lines up suspiciously well with one of those variables — rather than with any marker gene — is a warning sign, not a discovery. This is Agent-B checklist item 16: could a batch, lane, or chemistry effect explain a cluster boundary better than a real cell-type difference? Today's single shared sample has one lane pair (L001/L002) that were combined during alignment specifically so this isn't a live confound in your output — but the check is worth running as a habit, since it will matter the moment you work with real multi-batch data.

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "pct_counts_mt"])

If a cluster boundary tracks `total_counts` or `pct_counts_mt` more cleanly than it tracks any biology you'd expect, that's a QC artifact bleeding into your clustering, not a cell type.

## Save your checkpoint

[08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb) loads this back in.

In [ ]:
adata.write_h5ad("results/checkpoint_07_clustered.h5ad")
print("Saved to results/checkpoint_07_clustered.h5ad")


## Agent-assisted clustering, done right

> Weak: "Cluster this data."
>
> Strong: "Cluster this AnnData object with Leiden at two different resolutions, plot both on the UMAP, and tell me which cluster boundaries change between the two — that's where the resolution choice actually matters biologically, versus where it's stable."

## Practice

Run Agent-B checklist items 15 and 16 against your clustering code and plots above.

## Further reading

- [Single-cell best practices — Dimensionality Reduction](https://www.sc-best-practices.org/preprocessing_visualization/dimensionality_reduction.html)
- [Single-cell best practices — Clustering](https://www.sc-best-practices.org/cellular_structure/clustering.html)